# Use Case 05: Product Recommendation Engine

**CareerAlign GCP Machine Learning Engineer — Real-World Use Cases**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-mle/usecases/notebooks/05-recommendation-engine.ipynb)

---

This notebook builds a **hybrid product recommendation engine** from scratch using synthetic e-commerce data.
We implement collaborative filtering (matrix factorization via SVD), content-based filtering (TF-IDF on
product descriptions), and a hybrid scoring approach — the same techniques used in production GCP
recommendation systems with BigQuery ML and Vertex AI.

### What You Will Build
1. Generate synthetic e-commerce data (1,000 users, 500 products, 50,000 interactions)
2. Exploratory analysis of interaction patterns and sparsity
3. Collaborative filtering with scipy SVD (matrix factorization)
4. Content-based filtering with TF-IDF on product descriptions
5. Hybrid recommendation scoring
6. Cold-start handling strategies
7. Evaluation metrics: Precision@K, Recall@K, NDCG@K, coverage, diversity

### GCP Production Mapping
| This Notebook | GCP Production |
|---|---|
| scipy SVD | BigQuery ML `MATRIX_FACTORIZATION` |
| sklearn TF-IDF | Vertex AI custom training with embeddings |
| pandas DataFrames | BigQuery tables |
| In-memory dict | Vertex AI Feature Store |
| Manual scoring | Vertex AI Endpoint |

In [ ]:
# Install required packages
!pip install pandas numpy scikit-learn scipy matplotlib seaborn --quiet

print("All packages installed successfully.")

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print("Imports complete.")

---
## 1. Generate Synthetic E-Commerce Data

We create realistic synthetic data with:
- **1,000 users** with varying activity levels (power users, casual browsers, new users)
- **500 products** across 10 categories with descriptions and attributes
- **50,000 interaction events** (views, clicks, add-to-cart, purchases) with realistic distributions

In [ ]:
# --- Product Catalog ---
NUM_PRODUCTS = 500
NUM_USERS = 1000
NUM_INTERACTIONS = 50000

categories = [
    'Electronics', 'Clothing', 'Home & Kitchen', 'Books',
    'Sports & Outdoors', 'Beauty', 'Toys & Games', 'Grocery',
    'Automotive', 'Health'
]

brands_per_category = {
    'Electronics': ['TechPro', 'GadgetX', 'SmartLife', 'DigiMax', 'NovaTech'],
    'Clothing': ['UrbanWear', 'StyleFit', 'ComfortPlus', 'ActiveEdge', 'ClassicMode'],
    'Home & Kitchen': ['HomeChef', 'CozyLiving', 'KitchenPro', 'CleanSpace', 'DécorArts'],
    'Books': ['PageTurner', 'MindBooks', 'LearnPress', 'StoryHouse', 'AcadPublish'],
    'Sports & Outdoors': ['FitGear', 'TrailBlazer', 'ProSport', 'OutdoorX', 'AthleticEdge'],
    'Beauty': ['GlowUp', 'PureSkin', 'LuxBeauty', 'NaturalGlow', 'BeautyLab'],
    'Toys & Games': ['FunFactory', 'PlayTime', 'BrainGames', 'KidJoy', 'GameMaster'],
    'Grocery': ['FreshFarm', 'OrganicLife', 'DailyBasics', 'GourmetSelect', 'NutriPack'],
    'Automotive': ['AutoParts', 'DriveMax', 'CarCare', 'MotorPro', 'RoadReady'],
    'Health': ['WellLife', 'VitaBoost', 'HealthFirst', 'MediCare', 'FitWell'],
}

adjectives = ['Premium', 'Essential', 'Classic', 'Pro', 'Ultra', 'Smart', 'Advanced', 'Deluxe', 'Compact', 'Eco']
nouns = {
    'Electronics': ['Headphones', 'Charger', 'Speaker', 'Tablet Stand', 'USB Hub', 'Mouse', 'Keyboard', 'Monitor Light'],
    'Clothing': ['T-Shirt', 'Jacket', 'Jeans', 'Sneakers', 'Hoodie', 'Dress Shirt', 'Shorts', 'Sweater'],
    'Home & Kitchen': ['Blender', 'Pan Set', 'Knife Set', 'Coffee Maker', 'Toaster', 'Cutting Board', 'Mug Set', 'Spatula'],
    'Books': ['Guide', 'Handbook', 'Novel', 'Textbook', 'Workbook', 'Anthology', 'Manual', 'Encyclopedia'],
    'Sports & Outdoors': ['Yoga Mat', 'Water Bottle', 'Resistance Band', 'Jump Rope', 'Dumbbell', 'Backpack', 'Tent', 'Bike Light'],
    'Beauty': ['Moisturizer', 'Serum', 'Cleanser', 'Sunscreen', 'Lip Balm', 'Face Mask', 'Toner', 'Eye Cream'],
    'Toys & Games': ['Puzzle', 'Board Game', 'Action Figure', 'Building Set', 'Card Game', 'Stuffed Animal', 'RC Car', 'Art Kit'],
    'Grocery': ['Snack Mix', 'Granola Bar', 'Coffee Beans', 'Tea Set', 'Olive Oil', 'Spice Kit', 'Protein Bar', 'Dried Fruit'],
    'Automotive': ['Floor Mats', 'Phone Mount', 'Seat Cover', 'Air Freshener', 'Dash Cam', 'Tire Gauge', 'Wiper Blades', 'Jump Starter'],
    'Health': ['Vitamins', 'First Aid Kit', 'Thermometer', 'Blood Pressure Monitor', 'Supplements', 'Massage Gun', 'Pill Organizer', 'Scale'],
}

products = []
for i in range(NUM_PRODUCTS):
    cat = categories[i % len(categories)]
    brand = np.random.choice(brands_per_category[cat])
    adj = np.random.choice(adjectives)
    noun = np.random.choice(nouns[cat])
    title = f"{brand} {adj} {noun}"
    price = round(np.random.lognormal(mean=3.5, sigma=0.8), 2)
    price = max(5.0, min(price, 500.0))

    # Generate a synthetic description from category-relevant keywords
    desc_words = [cat.lower(), noun.lower(), adj.lower(), brand.lower(),
                  'quality', 'durable', 'popular', 'best-selling',
                  np.random.choice(['comfortable', 'lightweight', 'ergonomic', 'versatile', 'portable']),
                  np.random.choice(['great value', 'top rated', 'customer favorite', 'award winning'])]
    description = ' '.join(desc_words)

    products.append({
        'product_id': f'P{i:04d}',
        'title': title,
        'category': cat,
        'brand': brand,
        'price': price,
        'description': description,
    })

products_df = pd.DataFrame(products)
print(f"Product catalog: {len(products_df)} products across {products_df['category'].nunique()} categories")
print(f"\nCategory distribution:")
print(products_df['category'].value_counts())
print(f"\nPrice range: ${products_df['price'].min():.2f} - ${products_df['price'].max():.2f}")
print(f"Median price: ${products_df['price'].median():.2f}")
products_df.head(10)

In [ ]:
# --- User Profiles ---
# Users have preferred categories (1-3 primary interests) and activity levels

users = []
for i in range(NUM_USERS):
    # Each user has 1-3 preferred categories
    n_prefs = np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2])
    preferred_cats = list(np.random.choice(categories, size=n_prefs, replace=False))

    # Activity level: power (10%), regular (60%), casual (25%), new (5%)
    activity = np.random.choice(
        ['power', 'regular', 'casual', 'new'],
        p=[0.10, 0.60, 0.25, 0.05]
    )

    users.append({
        'user_id': f'U{i:04d}',
        'preferred_categories': preferred_cats,
        'activity_level': activity,
    })

users_df = pd.DataFrame(users)
print(f"Users: {len(users_df)}")
print(f"\nActivity level distribution:")
print(users_df['activity_level'].value_counts())

In [ ]:
# --- Generate Interaction Events ---
# Interactions are biased toward users' preferred categories

event_types = ['view', 'click', 'add_to_cart', 'purchase']
event_weights_map = {'view': 1, 'click': 2, 'add_to_cart': 3, 'purchase': 5}

# Activity level -> expected number of interactions
activity_interactions = {'power': 200, 'regular': 60, 'casual': 20, 'new': 3}

interactions = []
product_cats = products_df.set_index('product_id')['category'].to_dict()
all_product_ids = products_df['product_id'].tolist()

for _, user in users_df.iterrows():
    uid = user['user_id']
    prefs = user['preferred_categories']
    n_events = int(np.random.poisson(activity_interactions[user['activity_level']]))
    n_events = max(1, min(n_events, 500))  # clamp

    # Build product selection probabilities biased toward preferred categories
    probs = []
    for pid in all_product_ids:
        cat = product_cats[pid]
        if cat in prefs:
            probs.append(5.0)  # 5x more likely for preferred category
        else:
            probs.append(1.0)
    probs = np.array(probs)
    probs /= probs.sum()

    # Select products for this user
    selected_products = np.random.choice(all_product_ids, size=n_events, p=probs, replace=True)

    for pid in selected_products:
        # Event type distribution: views are most common, purchases rarest
        etype = np.random.choice(event_types, p=[0.50, 0.28, 0.14, 0.08])
        interactions.append({
            'user_id': uid,
            'product_id': pid,
            'event_type': etype,
        })

interactions_df = pd.DataFrame(interactions)
print(f"Total interactions generated: {len(interactions_df):,}")
print(f"\nEvent type distribution:")
print(interactions_df['event_type'].value_counts())
print(f"\nUnique users with interactions: {interactions_df['user_id'].nunique()}")
print(f"Unique products with interactions: {interactions_df['product_id'].nunique()}")

---
## 2. Exploratory Analysis

Understand the interaction patterns, sparsity, and distribution before building models.

In [ ]:
# Compute weighted interaction scores (user x product)
interactions_df['weight'] = interactions_df['event_type'].map(event_weights_map)

user_item_scores = (
    interactions_df
    .groupby(['user_id', 'product_id'])['weight']
    .sum()
    .reset_index()
    .rename(columns={'weight': 'score'})
)

print(f"User-item pairs: {len(user_item_scores):,}")
print(f"Sparsity: {1 - len(user_item_scores) / (NUM_USERS * NUM_PRODUCTS):.4%}")
print(f"\nScore distribution:")
print(user_item_scores['score'].describe())
user_item_scores.head(10)

In [ ]:
# Visualization: interaction distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Interactions per user
user_counts = interactions_df.groupby('user_id').size()
axes[0, 0].hist(user_counts, bins=50, color='#f59e0b', edgecolor='#0b0d14', alpha=0.85)
axes[0, 0].set_title('Interactions per User', fontweight='bold')
axes[0, 0].set_xlabel('Number of interactions')
axes[0, 0].set_ylabel('Number of users')
axes[0, 0].axvline(user_counts.median(), color='red', linestyle='--', label=f'Median: {user_counts.median():.0f}')
axes[0, 0].legend()

# 2. Interactions per product
product_counts = interactions_df.groupby('product_id').size()
axes[0, 1].hist(product_counts, bins=50, color='#3b82f6', edgecolor='#0b0d14', alpha=0.85)
axes[0, 1].set_title('Interactions per Product', fontweight='bold')
axes[0, 1].set_xlabel('Number of interactions')
axes[0, 1].set_ylabel('Number of products')
axes[0, 1].axvline(product_counts.median(), color='red', linestyle='--', label=f'Median: {product_counts.median():.0f}')
axes[0, 1].legend()

# 3. Event type distribution
event_counts = interactions_df['event_type'].value_counts()
colors = ['#8e94b5', '#3b82f6', '#f59e0b', '#22c55e']
axes[1, 0].bar(event_counts.index, event_counts.values, color=colors, edgecolor='#0b0d14')
axes[1, 0].set_title('Event Type Distribution', fontweight='bold')
axes[1, 0].set_ylabel('Count')
for i, (et, cnt) in enumerate(event_counts.items()):
    axes[1, 0].text(i, cnt + 200, f'{cnt:,}', ha='center', fontweight='bold')

# 4. Score distribution
axes[1, 1].hist(user_item_scores['score'], bins=40, color='#a855f7', edgecolor='#0b0d14', alpha=0.85)
axes[1, 1].set_title('Weighted Score Distribution', fontweight='bold')
axes[1, 1].set_xlabel('Aggregated score')
axes[1, 1].set_ylabel('Number of user-item pairs')

plt.tight_layout()
plt.suptitle('Exploratory Analysis: E-Commerce Interactions', y=1.02, fontsize=16, fontweight='bold')
plt.show()

In [ ]:
# Category interaction heatmap
interactions_with_cat = interactions_df.merge(
    products_df[['product_id', 'category']], on='product_id'
)

# Get user preferred category (first one)
user_pref = users_df.copy()
user_pref['primary_category'] = user_pref['preferred_categories'].apply(lambda x: x[0])

interactions_with_both = interactions_with_cat.merge(
    user_pref[['user_id', 'primary_category']], on='user_id'
)

# Cross-tab: user primary category vs product category
cross_tab = pd.crosstab(
    interactions_with_both['primary_category'],
    interactions_with_both['category'],
    normalize='index'
)

plt.figure(figsize=(12, 8))
sns.heatmap(cross_tab, annot=True, fmt='.2f', cmap='YlOrBr', linewidths=0.5)
plt.title('User Primary Interest vs Product Category Interaction Rate', fontweight='bold', fontsize=14)
plt.xlabel('Product Category')
plt.ylabel('User Primary Interest')
plt.tight_layout()
plt.show()

print("Diagonal dominance confirms users preferentially interact with their preferred categories.")

---
## 3. Collaborative Filtering — Matrix Factorization (SVD)

We build a user-item interaction matrix and decompose it using truncated SVD.
This is equivalent to what BigQuery ML does with `model_type='MATRIX_FACTORIZATION'`.

```sql
-- BigQuery ML equivalent
CREATE OR REPLACE MODEL recommendations.collab_filter_model
OPTIONS (
  model_type            = 'MATRIX_FACTORIZATION',
  user_col              = 'user_id',
  item_col              = 'product_id',
  rating_col            = 'interaction_score',
  feedback_type         = 'IMPLICIT',
  num_factors           = 50,
  l2_reg                = 0.1,
  wals_alpha            = 40,
  max_iterations        = 20
) AS
SELECT user_id, product_id, interaction_score
FROM recommendations.user_item_scores;
```

In [ ]:
# Build user-item matrix
user_ids = sorted(user_item_scores['user_id'].unique())
product_ids = sorted(user_item_scores['product_id'].unique())

user_id_map = {uid: idx for idx, uid in enumerate(user_ids)}
product_id_map = {pid: idx for idx, pid in enumerate(product_ids)}
reverse_user_map = {idx: uid for uid, idx in user_id_map.items()}
reverse_product_map = {idx: pid for pid, idx in product_id_map.items()}

rows = user_item_scores['user_id'].map(user_id_map).values
cols = user_item_scores['product_id'].map(product_id_map).values
vals = user_item_scores['score'].values.astype(np.float32)

n_users = len(user_ids)
n_products = len(product_ids)

# Create sparse matrix
interaction_matrix = csr_matrix((vals, (rows, cols)), shape=(n_users, n_products))

print(f"Interaction matrix shape: {interaction_matrix.shape}")
print(f"Non-zero entries: {interaction_matrix.nnz:,}")
print(f"Sparsity: {1 - interaction_matrix.nnz / (n_users * n_products):.4%}")
print(f"Density: {interaction_matrix.nnz / (n_users * n_products):.4%}")

In [ ]:
# Perform truncated SVD (matrix factorization)
# R ≈ U * Sigma * Vt
# num_factors = 50 latent dimensions
NUM_FACTORS = 50

print(f"Running SVD with {NUM_FACTORS} latent factors...")
U, sigma, Vt = svds(interaction_matrix.astype(float), k=NUM_FACTORS)

# Convert sigma to diagonal matrix
sigma_diag = np.diag(sigma)

# Predicted scores = U * Sigma * Vt
predicted_scores = U @ sigma_diag @ Vt

print(f"U shape (user factors): {U.shape}")
print(f"Sigma shape: {sigma_diag.shape}")
print(f"Vt shape (item factors): {Vt.shape}")
print(f"Predicted scores shape: {predicted_scores.shape}")
print(f"\nTop singular values: {sigma[-5:][::-1]}")

# Store as DataFrame for convenience
collab_scores_df = pd.DataFrame(
    predicted_scores,
    index=user_ids,
    columns=product_ids
)

print(f"\nSample predicted scores (first 5 users x 5 products):")
collab_scores_df.iloc[:5, :5]

In [ ]:
def get_collab_recommendations(user_id, top_k=10, exclude_seen=True):
    """Get top-K collaborative filtering recommendations for a user."""
    if user_id not in user_id_map:
        return pd.DataFrame()  # Cold-start user

    user_idx = user_id_map[user_id]
    scores = predicted_scores[user_idx]

    # Exclude already-interacted products
    if exclude_seen:
        seen_products = set(
            user_item_scores[user_item_scores['user_id'] == user_id]['product_id']
        )
        seen_indices = [product_id_map[pid] for pid in seen_products if pid in product_id_map]
        scores = scores.copy()
        scores[seen_indices] = -np.inf

    # Get top-K
    top_indices = np.argsort(scores)[::-1][:top_k]
    recs = []
    for idx in top_indices:
        pid = reverse_product_map[idx]
        recs.append({
            'product_id': pid,
            'collab_score': scores[idx],
        })

    return pd.DataFrame(recs).merge(products_df[['product_id', 'title', 'category', 'price']], on='product_id')

# Example: recommendations for a power user
sample_user = users_df[users_df['activity_level'] == 'power'].iloc[0]['user_id']
print(f"Collaborative filtering recommendations for {sample_user}:")
print(f"User preferred categories: {users_df[users_df['user_id'] == sample_user].iloc[0]['preferred_categories']}")
print()
collab_recs = get_collab_recommendations(sample_user, top_k=10)
collab_recs

---
## 4. Content-Based Filtering (TF-IDF)

Use product descriptions and attributes to compute item-item similarity.
Recommend products similar to what the user has previously liked.

```sql
-- BigQuery ML equivalent: use an autoencoder or text embedding model
-- for product embeddings, then compute cosine similarity
CREATE OR REPLACE MODEL recommendations.product_embedding_model
OPTIONS (
  model_type = 'AUTOENCODER',
  hidden_units = [256, 128, 64],
  activation_fn = 'RELU'
) AS
SELECT product_id, category, brand, price, description
FROM recommendations.product_catalog;
```

In [ ]:
# Build TF-IDF vectors from product descriptions + category + brand
products_df['combined_text'] = (
    products_df['description'] + ' ' +
    products_df['category'] + ' ' +
    products_df['brand']
)

tfidf = TfidfVectorizer(max_features=500, stop_words='english')
tfidf_matrix = tfidf.fit_transform(products_df['combined_text'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

# Compute item-item cosine similarity
item_similarity = cosine_similarity(tfidf_matrix)
print(f"Item similarity matrix shape: {item_similarity.shape}")

# Store as DataFrame
item_sim_df = pd.DataFrame(
    item_similarity,
    index=products_df['product_id'],
    columns=products_df['product_id']
)

print(f"\nSample similarities (first 5 products):")
item_sim_df.iloc[:5, :5]

In [ ]:
def get_content_recommendations(user_id, top_k=10, exclude_seen=True):
    """Get content-based recommendations by finding products similar to user's history."""
    # Get products the user interacted with, weighted by score
    user_history = user_item_scores[user_item_scores['user_id'] == user_id]

    if len(user_history) == 0:
        return pd.DataFrame()  # Cold-start user

    seen_products = set(user_history['product_id'])

    # Build user profile: weighted average of liked product TF-IDF vectors
    profile_indices = []
    profile_weights = []
    for _, row in user_history.iterrows():
        pid = row['product_id']
        idx = products_df[products_df['product_id'] == pid].index
        if len(idx) > 0:
            profile_indices.append(idx[0])
            profile_weights.append(row['score'])

    if not profile_indices:
        return pd.DataFrame()

    # Weighted average of TF-IDF vectors
    weights = np.array(profile_weights)
    weights = weights / weights.sum()
    user_profile_vec = np.zeros(tfidf_matrix.shape[1])
    for idx, w in zip(profile_indices, weights):
        user_profile_vec += w * tfidf_matrix[idx].toarray().flatten()

    # Score all products against user profile
    content_scores = cosine_similarity(
        user_profile_vec.reshape(1, -1),
        tfidf_matrix
    ).flatten()

    # Exclude seen products
    candidate_scores = []
    for i, pid in enumerate(products_df['product_id']):
        if exclude_seen and pid in seen_products:
            continue
        candidate_scores.append({'product_id': pid, 'content_score': content_scores[i]})

    recs_df = pd.DataFrame(candidate_scores).sort_values('content_score', ascending=False).head(top_k)
    return recs_df.merge(products_df[['product_id', 'title', 'category', 'price']], on='product_id')

# Example
print(f"Content-based recommendations for {sample_user}:")
print(f"User preferred categories: {users_df[users_df['user_id'] == sample_user].iloc[0]['preferred_categories']}")
print()
content_recs = get_content_recommendations(sample_user, top_k=10)
content_recs

---
## 5. Hybrid Recommendation Scoring

Combine collaborative and content-based scores with a tunable alpha parameter:

$$\text{hybrid\_score} = \alpha \times \text{collab\_score} + (1 - \alpha) \times \text{content\_score}$$

```sql
-- BigQuery equivalent: join collaborative and content scores
SELECT
  COALESCE(c.user_id, cb.user_id) AS user_id,
  COALESCE(c.product_id, cb.product_id) AS product_id,
  0.6 * COALESCE(c.collab_score, 0) +
  0.4 * COALESCE(cb.content_score, 0) AS hybrid_score
FROM collab_scores c
FULL OUTER JOIN content_scores cb
  ON c.user_id = cb.user_id AND c.product_id = cb.product_id
```

In [ ]:
def get_hybrid_recommendations(user_id, alpha=0.6, top_k=10):
    """Hybrid recommendations combining collaborative and content-based scores."""
    collab = get_collab_recommendations(user_id, top_k=50, exclude_seen=True)
    content = get_content_recommendations(user_id, top_k=50, exclude_seen=True)

    if collab.empty and content.empty:
        return pd.DataFrame()

    # Normalize scores to [0, 1]
    if not collab.empty and collab['collab_score'].max() > collab['collab_score'].min():
        collab['collab_norm'] = (
            (collab['collab_score'] - collab['collab_score'].min()) /
            (collab['collab_score'].max() - collab['collab_score'].min())
        )
    else:
        collab['collab_norm'] = 0.5

    if not content.empty and content['content_score'].max() > content['content_score'].min():
        content['content_norm'] = (
            (content['content_score'] - content['content_score'].min()) /
            (content['content_score'].max() - content['content_score'].min())
        )
    else:
        content['content_norm'] = 0.5

    # Merge on product_id
    merged = pd.merge(
        collab[['product_id', 'collab_norm']],
        content[['product_id', 'content_norm']],
        on='product_id',
        how='outer'
    ).fillna(0)

    # Compute hybrid score
    merged['hybrid_score'] = alpha * merged['collab_norm'] + (1 - alpha) * merged['content_norm']
    merged = merged.sort_values('hybrid_score', ascending=False).head(top_k)

    # Add product details
    return merged.merge(products_df[['product_id', 'title', 'category', 'price']], on='product_id')

# Example
print(f"Hybrid recommendations for {sample_user} (alpha=0.6):")
print(f"User preferred categories: {users_df[users_df['user_id'] == sample_user].iloc[0]['preferred_categories']}")
print()
hybrid_recs = get_hybrid_recommendations(sample_user, alpha=0.6, top_k=10)
hybrid_recs

In [ ]:
# Compare recommendation sources for the same user
print(f"=" * 70)
print(f"Comparison for user {sample_user}")
print(f"Preferred categories: {users_df[users_df['user_id'] == sample_user].iloc[0]['preferred_categories']}")
print(f"=" * 70)

for method_name, recs in [('Collaborative', collab_recs), ('Content-Based', content_recs), ('Hybrid', hybrid_recs)]:
    if recs.empty:
        print(f"\n{method_name}: No recommendations")
        continue
    cats = recs['category'].value_counts().to_dict()
    print(f"\n{method_name} — Category breakdown: {cats}")
    print(f"  Avg price: ${recs['price'].mean():.2f}")
    print(f"  Unique categories: {recs['category'].nunique()}")

---
## 6. Cold-Start Handling

Handle new users (no interaction history) and new products (no collaborative signal)
using fallback strategies.

In [ ]:
# --- Popularity-based fallback ---
# Global popularity: products ranked by total weighted interactions
product_popularity = (
    user_item_scores
    .groupby('product_id')['score']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'total_score', 'count': 'num_users'})
    .sort_values('total_score', ascending=False)
    .reset_index()
)

# Category-level popularity
product_popularity = product_popularity.merge(
    products_df[['product_id', 'category', 'title', 'price']], on='product_id'
)

print("Global Top 10 Most Popular Products:")
product_popularity.head(10)[['product_id', 'title', 'category', 'total_score', 'num_users']]

In [ ]:
def cold_start_recommendations(user_id=None, category_hint=None, top_k=10):
    """
    Cold-start recommendation strategy.
    - If no user history: use popularity (optionally filtered by category hint)
    - If 1-2 interactions: content-based only
    - If 3-5: content + lightweight KNN
    - If 5+: full hybrid
    """
    if user_id is None:
        interaction_count = 0
    else:
        interaction_count = len(user_item_scores[user_item_scores['user_id'] == user_id])

    if interaction_count == 0:
        strategy = 'popularity'
        if category_hint:
            recs = product_popularity[product_popularity['category'] == category_hint].head(top_k)
        else:
            recs = product_popularity.head(top_k)
        recs = recs[['product_id', 'title', 'category', 'price', 'total_score']].copy()
        recs['strategy'] = strategy

    elif interaction_count <= 2:
        strategy = 'content_only'
        recs = get_content_recommendations(user_id, top_k=top_k)
        recs['strategy'] = strategy

    elif interaction_count <= 5:
        strategy = 'content_plus_popularity'
        content = get_content_recommendations(user_id, top_k=top_k * 2)
        if not content.empty:
            # Blend with popularity
            content = content.merge(
                product_popularity[['product_id', 'total_score']],
                on='product_id', how='left'
            ).fillna(0)
            content['blended'] = 0.7 * content['content_score'] + 0.3 * (
                content['total_score'] / content['total_score'].max()
            )
            recs = content.sort_values('blended', ascending=False).head(top_k)
        else:
            recs = product_popularity.head(top_k).copy()
        recs['strategy'] = strategy

    else:
        strategy = 'full_hybrid'
        recs = get_hybrid_recommendations(user_id, alpha=0.6, top_k=top_k)
        recs['strategy'] = strategy

    return recs

# Test cold-start scenarios
print("=" * 60)
print("COLD-START SCENARIOS")
print("=" * 60)

# Scenario 1: Brand new user (no history)
print("\n--- Scenario 1: Brand new user (no history) ---")
new_user_recs = cold_start_recommendations(user_id=None, category_hint='Electronics', top_k=5)
print(f"Strategy: {new_user_recs['strategy'].iloc[0]}")
print(new_user_recs[['product_id', 'title', 'category']].to_string(index=False))

# Scenario 2: User with minimal history
casual_user = users_df[users_df['activity_level'] == 'new'].iloc[0]['user_id']
casual_count = len(user_item_scores[user_item_scores['user_id'] == casual_user])
print(f"\n--- Scenario 2: User {casual_user} ({casual_count} interactions) ---")
casual_recs = cold_start_recommendations(user_id=casual_user, top_k=5)
print(f"Strategy: {casual_recs['strategy'].iloc[0]}")
if 'title' in casual_recs.columns:
    print(casual_recs[['product_id', 'title', 'category']].to_string(index=False))

# Scenario 3: Active user (full hybrid)
print(f"\n--- Scenario 3: Power user {sample_user} ---")
power_count = len(user_item_scores[user_item_scores['user_id'] == sample_user])
power_recs = cold_start_recommendations(user_id=sample_user, top_k=5)
print(f"Strategy: {power_recs['strategy'].iloc[0]} ({power_count} interactions)")
if 'title' in power_recs.columns:
    print(power_recs[['product_id', 'title', 'category']].to_string(index=False))

In [ ]:
# --- New Product Cold-Start ---
# For a new product, find similar existing products and inherit their collaborative scores

def new_product_bootstrap(new_product_desc, new_product_category, top_k_similar=10):
    """
    Bootstrap a new product into the recommendation pool by finding similar existing products.
    """
    # Compute TF-IDF for the new product
    combined = new_product_desc + ' ' + new_product_category
    new_vec = tfidf.transform([combined])

    # Find most similar existing products
    similarities = cosine_similarity(new_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k_similar]

    similar_products = []
    for idx in top_indices:
        pid = products_df.iloc[idx]['product_id']
        similar_products.append({
            'product_id': pid,
            'title': products_df.iloc[idx]['title'],
            'category': products_df.iloc[idx]['category'],
            'similarity': similarities[idx],
        })

    return pd.DataFrame(similar_products)

# Example: bootstrap a new Electronics product
print("Bootstrapping new product: 'NovaTech Ultra Wireless Earbuds'")
print("Category: Electronics")
print()
similar = new_product_bootstrap(
    'novatech ultra wireless earbuds bluetooth premium quality portable',
    'Electronics',
    top_k_similar=10
)
print("Most similar existing products:")
similar

---
## 7. Evaluation Metrics

Evaluate recommendation quality using standard information retrieval metrics.
We use a **train/test split** approach: hold out 20% of interactions as the test set and
evaluate whether the model can predict them.

In [ ]:
# Train/test split: for each user, hold out 20% of interactions
from sklearn.model_selection import train_test_split

train_data = []
test_data = []

for uid, group in user_item_scores.groupby('user_id'):
    if len(group) < 5:
        # Not enough data to split — put all in train
        train_data.append(group)
        continue
    train_g, test_g = train_test_split(group, test_size=0.2, random_state=42)
    train_data.append(train_g)
    test_data.append(test_g)

train_scores = pd.concat(train_data, ignore_index=True)
test_scores = pd.concat(test_data, ignore_index=True) if test_data else pd.DataFrame()

print(f"Train set: {len(train_scores):,} user-item pairs")
print(f"Test set:  {len(test_scores):,} user-item pairs")
print(f"Test users: {test_scores['user_id'].nunique()}")

# Build ground truth: for each user, the set of products in the test set
ground_truth = defaultdict(set)
for _, row in test_scores.iterrows():
    ground_truth[row['user_id']].add(row['product_id'])

In [ ]:
# --- Evaluation functions ---

def precision_at_k(recommended, relevant, k):
    """Fraction of top-K recommendations that are relevant."""
    rec_set = set(recommended[:k])
    return len(rec_set & relevant) / k if k > 0 else 0.0

def recall_at_k(recommended, relevant, k):
    """Fraction of relevant items that appear in top-K."""
    rec_set = set(recommended[:k])
    return len(rec_set & relevant) / len(relevant) if len(relevant) > 0 else 0.0

def ndcg_at_k(recommended, relevant, k):
    """Normalized Discounted Cumulative Gain at K."""
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)  # i+2 because log2(1)=0

    # Ideal DCG
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))

    return dcg / idcg if idcg > 0 else 0.0

def catalog_coverage(all_recommendations, total_products):
    """Fraction of total catalog that appears in any recommendation."""
    all_recommended = set()
    for recs in all_recommendations:
        all_recommended.update(recs)
    return len(all_recommended) / total_products

def intra_list_diversity(recommended_products, similarity_matrix, product_id_list):
    """Average dissimilarity among recommended items."""
    if len(recommended_products) < 2:
        return 0.0

    pid_to_idx = {pid: idx for idx, pid in enumerate(product_id_list)}
    indices = [pid_to_idx[pid] for pid in recommended_products if pid in pid_to_idx]

    if len(indices) < 2:
        return 0.0

    total_dissim = 0.0
    count = 0
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            total_dissim += (1.0 - similarity_matrix[indices[i], indices[j]])
            count += 1

    return total_dissim / count if count > 0 else 0.0

print("Evaluation functions defined.")

In [ ]:
# --- Evaluate all methods ---
K = 10
eval_users = [uid for uid in ground_truth if len(ground_truth[uid]) >= 2]
print(f"Evaluating on {len(eval_users)} users with >= 2 test items...")

# Methods to evaluate
methods = {
    'Popularity': lambda uid: product_popularity['product_id'].tolist()[:K],
    'Collaborative': lambda uid: get_collab_recommendations(uid, top_k=K)['product_id'].tolist() if not get_collab_recommendations(uid, top_k=K).empty else [],
    'Content-Based': lambda uid: get_content_recommendations(uid, top_k=K)['product_id'].tolist() if not get_content_recommendations(uid, top_k=K).empty else [],
    'Hybrid (a=0.6)': lambda uid: get_hybrid_recommendations(uid, alpha=0.6, top_k=K)['product_id'].tolist() if not get_hybrid_recommendations(uid, alpha=0.6, top_k=K).empty else [],
}

results = {}
all_recs_per_method = defaultdict(list)

for method_name, rec_fn in methods.items():
    p_scores = []
    r_scores = []
    n_scores = []
    div_scores = []

    for uid in eval_users[:200]:  # Sample 200 users for speed
        recs = rec_fn(uid)
        relevant = ground_truth[uid]

        if not recs:
            continue

        p_scores.append(precision_at_k(recs, relevant, K))
        r_scores.append(recall_at_k(recs, relevant, K))
        n_scores.append(ndcg_at_k(recs, relevant, K))
        div_scores.append(
            intra_list_diversity(recs, item_similarity, products_df['product_id'].tolist())
        )
        all_recs_per_method[method_name].append(recs)

    coverage = catalog_coverage(
        all_recs_per_method[method_name],
        NUM_PRODUCTS
    )

    results[method_name] = {
        f'Precision@{K}': np.mean(p_scores) if p_scores else 0,
        f'Recall@{K}': np.mean(r_scores) if r_scores else 0,
        f'NDCG@{K}': np.mean(n_scores) if n_scores else 0,
        'Coverage': coverage,
        'Diversity': np.mean(div_scores) if div_scores else 0,
    }
    print(f"  {method_name}: done ({len(p_scores)} users evaluated)")

results_df = pd.DataFrame(results).T
print(f"\n{'=' * 70}")
print(f"EVALUATION RESULTS (K={K})")
print(f"{'=' * 70}")
results_df

In [ ]:
# Visualization of evaluation results
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

colors = ['#8e94b5', '#3b82f6', '#06b6d4', '#f59e0b']

for i, metric in enumerate(results_df.columns):
    bars = axes[i].bar(
        range(len(results_df)),
        results_df[metric],
        color=colors,
        edgecolor='#0b0d14',
        width=0.6
    )
    axes[i].set_title(metric, fontweight='bold', fontsize=12)
    axes[i].set_xticks(range(len(results_df)))
    axes[i].set_xticklabels(results_df.index, rotation=30, ha='right', fontsize=9)

    for bar, val in zip(bars, results_df[metric]):
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=9, fontweight='bold'
        )

plt.tight_layout()
plt.suptitle('Recommendation Method Comparison', y=1.04, fontsize=16, fontweight='bold')
plt.show()

In [ ]:
# --- Alpha sensitivity analysis ---
# How does the hybrid weighting parameter affect results?

alphas = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
alpha_results = []

sample_eval_users = eval_users[:100]  # Smaller sample for speed

for alpha in alphas:
    p_scores = []
    n_scores = []

    for uid in sample_eval_users:
        recs = get_hybrid_recommendations(uid, alpha=alpha, top_k=K)
        if recs.empty:
            continue
        rec_list = recs['product_id'].tolist()
        relevant = ground_truth[uid]
        p_scores.append(precision_at_k(rec_list, relevant, K))
        n_scores.append(ndcg_at_k(rec_list, relevant, K))

    alpha_results.append({
        'alpha': alpha,
        f'Precision@{K}': np.mean(p_scores) if p_scores else 0,
        f'NDCG@{K}': np.mean(n_scores) if n_scores else 0,
    })
    print(f"  alpha={alpha:.1f}: P@{K}={alpha_results[-1][f'Precision@{K}']:.4f}, NDCG@{K}={alpha_results[-1][f'NDCG@{K}']:.4f}")

alpha_df = pd.DataFrame(alpha_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(alpha_df['alpha'], alpha_df[f'Precision@{K}'], 'o-', color='#3b82f6', linewidth=2, label=f'Precision@{K}')
ax.plot(alpha_df['alpha'], alpha_df[f'NDCG@{K}'], 's-', color='#f59e0b', linewidth=2, label=f'NDCG@{K}')
ax.set_xlabel('Alpha (Collaborative Weight)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Hybrid Alpha Sensitivity Analysis', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(alphas)
plt.tight_layout()
plt.show()

best_alpha = alpha_df.loc[alpha_df[f'NDCG@{K}'].idxmax(), 'alpha']
print(f"\nBest alpha by NDCG@{K}: {best_alpha}")

---
## 8. Diversity Re-Ranking (MMR)

Apply Maximal Marginal Relevance to diversify the final recommendation list.

In [ ]:
def mmr_rerank(candidate_pids, candidate_scores, similarity_matrix, product_id_list,
               lambda_param=0.7, top_k=10):
    """
    Maximal Marginal Relevance re-ranking for diversity.
    MMR(i) = lambda * score(i) - (1 - lambda) * max_j_in_S similarity(i, j)
    """
    pid_to_idx = {pid: idx for idx, pid in enumerate(product_id_list)}

    # Normalize scores
    scores = np.array(candidate_scores)
    if scores.max() > scores.min():
        scores = (scores - scores.min()) / (scores.max() - scores.min())

    selected = []
    candidates = list(range(len(candidate_pids)))

    for _ in range(min(top_k, len(candidates))):
        best_mmr = -np.inf
        best_idx = None

        for c_idx in candidates:
            relevance = scores[c_idx]

            if selected:
                c_pid = candidate_pids[c_idx]
                c_sim_idx = pid_to_idx.get(c_pid)
                if c_sim_idx is not None:
                    max_sim = max(
                        similarity_matrix[
                            c_sim_idx,
                            pid_to_idx.get(candidate_pids[s], 0)
                        ]
                        for s in selected
                    )
                else:
                    max_sim = 0
            else:
                max_sim = 0

            mmr = lambda_param * relevance - (1 - lambda_param) * max_sim

            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = c_idx

        if best_idx is not None:
            selected.append(best_idx)
            candidates.remove(best_idx)

    return [candidate_pids[i] for i in selected]


# Compare: standard hybrid vs MMR-reranked
hybrid_full = get_hybrid_recommendations(sample_user, alpha=0.6, top_k=20)
if not hybrid_full.empty:
    standard_recs = hybrid_full['product_id'].tolist()[:10]
    mmr_recs = mmr_rerank(
        hybrid_full['product_id'].tolist(),
        hybrid_full['hybrid_score'].tolist(),
        item_similarity,
        products_df['product_id'].tolist(),
        lambda_param=0.6,
        top_k=10
    )

    print(f"Standard hybrid for {sample_user}:")
    standard_df = products_df[products_df['product_id'].isin(standard_recs)][['product_id', 'title', 'category']]
    print(f"  Categories: {standard_df['category'].value_counts().to_dict()}")
    print(f"  Diversity: {intra_list_diversity(standard_recs, item_similarity, products_df['product_id'].tolist()):.4f}")

    print(f"\nMMR-reranked (lambda=0.6):")
    mmr_df = products_df[products_df['product_id'].isin(mmr_recs)][['product_id', 'title', 'category']]
    print(f"  Categories: {mmr_df['category'].value_counts().to_dict()}")
    print(f"  Diversity: {intra_list_diversity(mmr_recs, item_similarity, products_df['product_id'].tolist()):.4f}")

---
## 9. Summary & GCP Production Notes

### What We Built
1. **Synthetic e-commerce dataset** with realistic user-product interaction patterns
2. **Collaborative filtering** using matrix factorization (SVD) to discover latent taste patterns
3. **Content-based filtering** using TF-IDF product embeddings for item similarity
4. **Hybrid scoring** that combines both approaches with a tunable alpha parameter
5. **Cold-start strategies** for new users and new products
6. **Comprehensive evaluation** with Precision@K, Recall@K, NDCG@K, coverage, and diversity
7. **MMR re-ranking** for recommendation diversity

### GCP Production Deployment

| Step | GCP Service | This Notebook |
|---|---|---|
| Data storage | BigQuery | pandas DataFrames |
| Collaborative filtering | BigQuery ML `MATRIX_FACTORIZATION` | scipy `svds` |
| Content embeddings | Vertex AI custom training / AutoEncoder | sklearn TF-IDF |
| Feature serving | Vertex AI Feature Store | Python dict |
| Model serving | Vertex AI Endpoints | Python functions |
| A/B testing | Vertex AI Experiments + BigQuery | Manual comparison |
| Orchestration | Vertex AI Pipelines | Sequential cells |
| Monitoring | Cloud Monitoring + Model Monitoring | Print statements |

### Key Takeaways
- **Hybrid > single method**: Combining collaborative and content-based filtering consistently outperforms either alone
- **Cold-start is critical**: Without fallback strategies, 5-10% of users get no recommendations
- **Diversity matters**: Pure relevance optimization leads to homogeneous lists; MMR re-ranking improves user experience
- **Alpha tuning**: The optimal collaborative/content balance depends on data density and user activity
- **Offline metrics are directional**: Always validate with online A/B tests before production deployment

In [ ]:
# Final summary statistics
print("=" * 60)
print("RECOMMENDATION ENGINE — FINAL SUMMARY")
print("=" * 60)
print(f"\nDataset:")
print(f"  Users:        {NUM_USERS:,}")
print(f"  Products:     {NUM_PRODUCTS:,}")
print(f"  Interactions: {len(interactions_df):,}")
print(f"  Sparsity:     {1 - len(user_item_scores) / (NUM_USERS * NUM_PRODUCTS):.2%}")
print(f"\nModel:")
print(f"  SVD factors:  {NUM_FACTORS}")
print(f"  TF-IDF features: {tfidf_matrix.shape[1]}")
print(f"\nBest Results (Hybrid):")
if 'Hybrid (a=0.6)' in results:
    for metric, value in results['Hybrid (a=0.6)'].items():
        print(f"  {metric:20s} {value:.4f}")
print(f"\nNotebook complete.")